# Source Data Discovery

## Objective

Profile the source retail inventory dataset before designing the integration pipeline.

This analysis establishes the dataset's grain, cardinality, completeness, key uniqueness, data types, and initial data-quality observations.

## 1. Load Source Data

In [21]:
import pandas as pd


df = pd.read_csv("../data/raw/retail_store_inventory.csv")

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## 2. Dataset Structure

Review the size of the dataset and cardinality of key business entities.

In [3]:
df.shape

(73100, 15)

In [4]:
df["Store ID"].nunique()

5

In [5]:
df["Product ID"].nunique()

20

## 3. Grain and Key Validation

Determine the grain of the source data and test whether Date + Store ID + Product ID uniquely identifies each record.

In [8]:
df["Date"].min(),df["Date"].max()

('2022-01-01', '2024-01-01')

In [9]:
df.duplicated(subset=["Date", "Store ID", "Product ID"]).sum()

np.int64(0)

In [10]:
df["Date"].nunique()

731

In [11]:
df.groupby("Date").size().describe()

count    731.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64

The source dataset contains a complete daily panel of 5 stores and 20 products across 731 unique dates. Each row represents one product at one store on one date. The composite key Date + Store ID + Product ID uniquely identifies each record, with exactly 100 Store/Product observations per date and no duplicate composite keys identified.

## 4. Missing Values and Data Types

Evaluate source completeness and inspect whether fields are represented using appropriate data types.

In [12]:
df.isnull().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forecast       0
Price                 0
Discount              0
Weather Condition     0
Holiday/Promotion     0
Competitor Pricing    0
Seasonality           0
dtype: int64

In [13]:
df.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition      object
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality            object
dtype: object

## 5. Numeric Data Profiling

Review distributions and ranges for numeric operational fields to identify potential anomalies requiring further investigation.

In [14]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077
std,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408
min,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000
25%,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000


## 6. Initial Data Quality Investigation

Investigate values that may conflict with expected business behavior before defining validation rules.

In [15]:
df[df["Demand Forecast"] < 0]

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
63,2022-01-01,S004,P0004,Groceries,West,437,0,160,-2.40,87.23,10,Sunny,0,90.36,Spring
141,2022-01-02,S003,P0002,Groceries,East,175,2,140,-3.40,87.50,0,Rainy,0,91.20,Autumn
278,2022-01-03,S004,P0019,Toys,North,140,1,47,-3.91,80.41,0,Snowy,0,76.90,Winter
511,2022-01-06,S001,P0012,Groceries,North,59,1,88,-8.37,63.21,0,Rainy,0,60.48,Spring
730,2022-01-08,S002,P0011,Electronics,South,64,7,79,-2.99,92.43,0,Snowy,0,88.55,Spring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72716,2023-12-29,S001,P0017,Clothing,North,487,7,132,-0.73,55.34,0,Sunny,1,58.03,Summer
72722,2023-12-29,S002,P0003,Toys,West,199,4,180,-5.50,82.08,10,Cloudy,1,77.99,Spring
72859,2023-12-30,S003,P0020,Toys,North,203,1,119,-7.68,94.33,5,Sunny,1,96.79,Autumn
73026,2024-01-01,S002,P0007,Toys,West,53,2,40,-6.08,63.66,20,Snowy,1,62.37,Autumn


In [16]:
(df["Demand Forecast"] < 0).sum()

np.int64(673)

In [18]:
display(df[df["Demand Forecast"] < 0]["Store ID"].value_counts())
display(df[df["Demand Forecast"] < 0]["Product ID"].value_counts())
display(df[df["Demand Forecast"] < 0]["Category"].value_counts())

Store ID
S003    143
S001    141
S002    137
S005    132
S004    120
Name: count, dtype: int64

Product ID
P0019    50
P0015    49
P0010    38
P0004    37
P0003    37
P0001    37
P0007    37
P0020    35
P0011    34
P0002    33
P0008    33
P0012    32
P0013    32
P0017    32
P0009    32
P0014    30
P0018    28
P0005    27
P0006    24
P0016    16
Name: count, dtype: int64

Category
Groceries      143
Toys           140
Clothing       132
Electronics    129
Furniture      129
Name: count, dtype: int64

673 records (0.92%) contain negative demand forecasts. The records are distributed across all stores, product IDs, and categories, suggesting the issue is not obviously isolated to a particular store or product segment. Further clarification of the forecast-generation methodology and expected business rules is required before treating these values as invalid.

In [26]:
(df["Units Sold"] > df["Inventory Level"]).sum()

np.int64(0)

In [29]:
(df["Inventory Level"] == 0).sum()

np.int64(0)

In [30]:
(df["Units Sold"] == 0).sum()

np.int64(360)

## 7. Source Discovery Findings

The initial source profiling established the following:

- The source contains **73,100 records across 15 attributes**.
- The dataset covers **5 stores, 20 products, and 731 unique dates**.
- Each date contains exactly **100 Store/Product observations**.
- One row represents the daily operational state of one product at one store.
- `Date + Store ID + Product ID` uniquely identifies each record.
- No missing values or duplicate composite keys were identified.
- No records report `Units Sold` greater than `Inventory Level`.
- No zero-inventory records are present in the source dataset.
- **360 records** report zero units sold.
- **673 records** contain negative `Demand Forecast` values.
- Negative demand forecasts are distributed across stores, products, and categories rather than being obviously isolated to one segment.
- The negative forecast values require business-rule clarification before determining whether they should be rejected, transformed, or retained.